# Decision medallion -- DCA barrier Gold materialization

Sprint 26 WS-B (issue #335). Deterministically materialises the Decision-tier
barrier Gold Delta table that turns a flat discharge-blocked candidate feed into
a ranked list of systemic barriers (design spec 3.3 item 3 / beat 3):

- `gold.fact_discharge_barrier` -- one row per systemic barrier (the DCA
  'N candidates collapse into M barriers' pattern), ranked by bed impact then
  age, with owner role / candidate count / clear time / spanning wards.

Synthetic + deterministic, no PHI (ADR-0013 / ADR-0016): candidates carry only
an opaque `candidate_key` + ontology ward IDs, so the Gold rows carry only
aggregate counts + ward IDs. **No LLM-guessed numbers** (design D2/D4): the
collapse/rank logic is the pure `derive_barriers` builder, reused verbatim and
unit-tested offline under `data-platform/decision/barriers/tests/`.

Grounds the reference ontology `hcp:Barrier` ICE (`barrierForWard` -> `hcp:Ward`)
and conforms to the `DC-DISCHARGE-BARRIER-v1` contract.

In [ ]:
import sys

# Notebook resources mirror the `data-platform/decision` tree under builtin/ in
# Fabric; adding the decision root to sys.path makes `barriers` importable so the
# gold builder can reuse the pure `derive_barriers` collapse/rank logic.
if "builtin/decision" not in sys.path:
    sys.path.insert(0, "builtin/decision")

from barriers import build_gold_barrier as bar


In [ ]:
# Barrier Gold table from the deterministic DEFAULT_CANDIDATES feed. Swap
# DEFAULT_CANDIDATES (or pass a real discharge-candidate reader) here to plug in
# the live candidate feed -- the D2 seam. Contract + ontology binding stay unchanged.
bar.run()


In [ ]:
# Inline verification -- count (5 barriers from the 8-candidate default feed) and
# the ranked barrier board for the evidence doc.
print("gold.fact_discharge_barrier", spark.table("gold.fact_discharge_barrier").count())

display(
    spark.table("gold.fact_discharge_barrier")
    .select("rank", "barrierType", "ownerRole", "candidateCount", "bedImpact", "agedH", "clearsAt", "wards")
    .orderBy("rank")
)
